# 딥러닝 2일차 (2) — 행렬 곱으로 신경망 계산 · 출력층 설계 (항등 · 시그모이드 · 소프트맥스)

수업 필기 `퍼셉트론`(소프트맥스 부분)과 메모 `딥러닝_1일_수정본` 을 교재에 맞춰 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `딥러닝` 교재8 p.31~40 — 다차원 배열의 계산 · 신경망 층과 행렬 곱 · 출력층의 역할 · 항등 / 시그모이드 / 소프트맥스 · 출력층 뉴런 수 · 출력층 설계 실습)

- 활성화 함수는 [`activation_functions.ipynb`](./activation_functions.ipynb) 에 있습니다.

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 행렬 곱 — `np.dot` (교재 p.31)

$(m, k) \times (k, n) = (m, n)$ — **앞 행렬의 열 수 = 뒤 행렬의 행 수**여야 곱할 수 있다. 결과의 (i, j) 칸 = 앞 행렬 i 번째 **행** · 뒤 행렬 j 번째 **열**의 내적.

In [2]:
import numpy as np

A = np.array([[1, 0],
              [2, 3]])
B = np.array([[2, 1],
              [3, 1]])

print("A · B =\n", np.dot(A, B))          # 교재 p.31 : [[2, 1], [13, 5]]
print("A @ B 도 같다 =\n", A @ B)
print("(1×2)+(0×3) =", 1 * 2 + 0 * 3, " / (2×2)+(3×3) =", 2 * 2 + 3 * 3)

print("\nA * B (원소끼리 곱 — 행렬 곱이 아님) =\n", A * B)

A · B =
 [[ 2  1]
 [13  5]]
A @ B 도 같다 =
 [[ 2  1]
 [13  5]]
(1×2)+(0×3) = 2  / (2×2)+(3×3) = 13

A * B (원소끼리 곱 — 행렬 곱이 아님) =
 [[2 0]
 [6 3]]


In [3]:
# 모양(shape) 규칙 : (10, 20) × (20, 5) = (10, 5)
P = np.ones((10, 20)); Q = np.ones((20, 5))
print(P.shape, "×", Q.shape, "=", (P @ Q).shape)

# 안쪽 숫자가 다르면 곱할 수 없다
try:
    np.ones((2, 3)) @ np.ones((2, 3))
except ValueError as e:
    print("모양 에러 :", e)

(10, 20) × (20, 5) = (10, 5)
모양 에러 : matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


## 2. 신경망의 한 층 = 행렬 곱 한 번 (교재 p.32)

뉴런 3개짜리 층에 입력 2개가 들어가면, 뉴런마다 $z_j = w_{1j}x_1 + w_{2j}x_2 + b_j$ 를 따로 계산해야 한다. 가중치를 **행렬 W (2 × 3)** 로 묶으면 한 번에 끝난다.

$$Y = X \cdot W + B \qquad (1, 2) \times (2, 3) + (3,) = (1, 3)$$

In [4]:
X = np.array([1.0, 0.5])                    # 입력 (2,)  — 특성 2개
W1 = np.array([[0.1, 0.3, 0.5],             # (2, 3) : 행 = 입력 2개, 열 = 뉴런 3개
               [0.2, 0.4, 0.6]])
B1 = np.array([0.1, 0.2, 0.3])              # (3,)  : 뉴런마다 편향 1개

Z1 = X @ W1 + B1                            # 3개 뉴런의 가중합을 한 번에
print("X·W1 + B1 =", Z1)

# 뉴런 하나씩 for 문으로 계산해도 같다
print("하나씩     =", [round(float(X[0] * W1[0, j] + X[1] * W1[1, j] + B1[j]), 2) for j in range(3)])

X·W1 + B1 = [0.3 0.7 1.1]
하나씩     = [0.3, 0.7, 1.1]


![행렬 곱 애니메이션](./images/anim_04_matrix_layer.gif)

🎬 왼쪽은 입력 2개 → 뉴런 3개 신경망, 오른쪽은 같은 계산을 행렬로 쓴 것. 뉴런 하나를 계산할 때마다 **W 의 열 하나**와 X 의 내적 + 편향이 그 뉴런의 z 가 된다. 행렬 곱 한 번 = 층 전체 계산.

## 3. 3층 신경망 순전파(forward) — 입력 → 은닉층 2개 → 출력층

| 층 | 모양 | 활성화 함수 |
|---|---|---|
| 입력층 | 2 | — |
| 은닉층 1 | 2 → 3 | 시그모이드 |
| 은닉층 2 | 3 → 2 | 시그모이드 |
| 출력층 | 2 → 2 | **항등 함수** (회귀 예시) |

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def identity_function(x):           # 항등 함수 : 입력 그대로 출력 (회귀 출력층)
    return x

network = {
    'W1': W1, 'b1': B1,
    'W2': np.array([[0.1, 0.4], [0.2, 0.5], [0.3, 0.6]]), 'b2': np.array([0.1, 0.2]),   # (3, 2)
    'W3': np.array([[0.1, 0.3], [0.2, 0.4]]),             'b3': np.array([0.1, 0.2]),   # (2, 2)
}

def forward(network, x):
    a1 = sigmoid(x @ network['W1'] + network['b1'])          # 은닉층 1
    a2 = sigmoid(a1 @ network['W2'] + network['b2'])         # 은닉층 2
    y = identity_function(a2 @ network['W3'] + network['b3'])  # 출력층
    return y

print("입력 (1.0, 0.5) →", forward(network, np.array([1.0, 0.5])).round(4))

# 입력 여러 개(배치)도 같은 코드 : (4, 2) → (4, 2)
X_batch = np.array([[1.0, 0.5], [0.0, 0.0], [1.0, 1.0], [-1.0, 2.0]])
print("배치 4개 :\n", forward(network, X_batch).round(4))

입력 (1.0, 0.5) → [0.3168 0.6963]
배치 4개 :
 [[0.3168 0.6963]
 [0.3081 0.6768]
 [0.3196 0.7025]
 [0.3167 0.6959]]


행(데이터) 수가 늘어도 **코드는 그대로** — $(4, 2) \times (2, 3) = (4, 3)$ 처럼 앞쪽 차원만 커진다. GPU 가 딥러닝에 쓰이는 이유가 이 **대량의 행렬 곱**을 병렬로 빨리 하기 때문.

## 4. 출력층 — 문제 유형에 맞는 함수 (교재 p.34~38)

| 문제 | 예 | 출력층 함수 | 출력 |
|---|---|---|---|
| **회귀** | 집값, 기온 | **항등 함수** $y = x$ | 숫자 그대로 |
| **이진 분류** | 스팸 / 정상, 이탈 / 유지 | **시그모이드** | 1 일 확률 p (0 일 확률 = 1 − p) |
| **다중 분류** | 붓꽃 3품종, 숫자 0~9 | **소프트맥스** | 클래스별 확률, **합 = 1** |

### 소프트맥스

$$y_k = \frac{e^{z_k}}{\sum_{i=1}^{n} e^{z_i}}$$

분자 = 그 클래스의 $e^{z_k}$, 분모 = 모든 클래스의 $e^{z_i}$ 합 → 각 출력은 0 ~ 1, **모두 더하면 1(100%)**.

In [6]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    exp_z = np.exp(z - np.max(z))      # 가장 큰 값을 빼고 지수 계산 (오버플로 방지, 결과는 같다)
    return exp_z / np.sum(exp_z)

z = np.array([0.3, 2.9, 4.0])          # 출력 뉴런 3개의 가중합 (필기 예시)
y = softmax(z)
print("소프트맥스 :", y.round(4), " 합 :", y.sum())
print("가장 큰 확률의 클래스 :", np.argmax(y), "  가장 큰 z 의 위치 :", np.argmax(z), " ← 순서는 그대로")

소프트맥스 : [0.0182 0.2452 0.7366]  합 : 1.0
가장 큰 확률의 클래스 : 2   가장 큰 z 의 위치 : 2  ← 순서는 그대로


In [7]:
# 왜 최댓값을 빼나? — z 가 크면 e^z 가 컴퓨터가 표현할 수 있는 수를 넘는다
big = np.array([1010, 1000, 990])
with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(big) / np.sum(np.exp(big))
print("그냥 계산      :", naive)                 # [nan nan nan]
print("최댓값 빼고 계산:", softmax(big).round(6))
with np.errstate(over='ignore'):
    print("np.exp(1010) =", np.exp(1010.0), " ← float64 로 표현할 수 있는 최대(약 1.8e308)를 넘었다")

그냥 계산      : [nan nan nan]
최댓값 빼고 계산: [9.99955e-01 4.50000e-05 0.00000e+00]
np.exp(1010) = inf  ← float64 로 표현할 수 있는 최대(약 1.8e308)를 넘었다


![소프트맥스 애니메이션](./images/anim_05_softmax.gif)

🎬 세 번째 뉴런의 가중합 $z_3$ 만 −2 → 6 으로 키웠다. $z_3$ 가 커질수록 그 클래스의 확률이 올라가고 **나머지 둘은 같은 비율로 줄어든다** — 확률의 합은 언제나 1. 지수 함수라서 z 의 작은 차이가 확률에서는 크게 벌어진다.

> **📝 고친 부분** (`퍼셉트론` 소프트맥스)
> - `np.exp(arr)` 를 그대로 계산 → z 가 수백 이상이면 **오버플로**로 `inf / inf = nan`. 최댓값을 빼고 계산(결과는 수학적으로 같다)
> - 셀 출력이 `array([...])` 로 저장돼 있었지만 코드는 `print(soft_num)` — 코드를 바꾼 뒤 다시 실행하지 않은 흔적
> - 주석 오타 `가각` → 각각, `뉴련` → 뉴런, `output_pert_sum` → 출력층의 가중합(z)

## 5. 출력층의 뉴런 수 정하기 (교재 p.39~40)

| 문제 | 출력 뉴런 수 | 함수 |
|---|---|---|
| 이진 분류 (스팸 판별) | **1** ('스팸일 확률' 하나) | 시그모이드 |
| 다중 분류 (붓꽃 3품종 · MNIST 0~9) | **클래스 수** (3 · 10) | 소프트맥스 |
| 회귀 (주택 가격) | **1** (가격 하나) | 항등 함수 |

### 출력층 설계 실습 — scikit-learn MLP 로 확인

`MLPClassifier` · `MLPRegressor` 는 **정답을 보고 출력층을 자동으로 설계**한다. 실제로 몇 개 · 어떤 함수를 골랐는지 확인해 보자.

In [8]:
import warnings
import pandas as pd
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=UserWarning)

iris = pd.read_csv('./iris_dataset.csv')
X_iris, y_iris = iris.drop(columns=['label']), iris['label']
house = pd.read_csv('./house_price.csv')
X_house, y_house = house.drop(columns=['medv']), house['medv']

tasks = {
    '다중 분류 (붓꽃 3품종)': (MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=0), X_iris, y_iris),
    '이진 분류 (setosa 인가?)': (MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=0), X_iris, (y_iris == 'setosa').astype(int)),
    '회귀 (보스턴 집값)': (MLPRegressor(hidden_layer_sizes=(32,), max_iter=3000, random_state=0), X_house, y_house),
}

rows = []
for name, (model, X, y) in tasks.items():
    pipe = make_pipeline(StandardScaler(), model).fit(X, y)      # 신경망도 스케일링 필요 (거리 · 기울기 기반)
    m = pipe[-1]
    rows.append({'문제': name, '은닉층 활성화': m.activation, '출력층 함수': m.out_activation_,
                 '출력 뉴런 수': m.n_outputs_, '층별 가중치 모양': [w.shape for w in m.coefs_]})
pd.DataFrame(rows)

,문제,은닉층 활성화,출력층 함수,출력 뉴런 수,층별 가중치 모양
0,다중 분류 (붓꽃 3품종),relu,softmax,3,"[(4, 8), (8, 3)]"
1,이진 분류 (setosa 인가?),relu,logistic,1,"[(4, 8), (8, 1)]"
2,회귀 (보스턴 집값),relu,identity,1,"[(13, 32), (32, 1)]"


교재의 설계 규칙 그대로 — 다중 분류는 **softmax · 3개**, 이진 분류는 **logistic(= 시그모이드) · 1개**, 회귀는 **identity(항등) · 1개**. 은닉층은 기본값이 **ReLU** 다.

### 직접 계산 = `predict_proba` — 행렬 곱 + ReLU + 소프트맥스

In [9]:
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=0, stratify=y_iris)
clf = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=0)).fit(X_tr, y_tr)
mlp = clf[-1]
print("평가용 정확도 :", clf.score(X_te, y_te))

# 학습된 가중치로 순전파를 numpy 로 직접
x = clf[0].transform(X_te[:3])                               # 스케일링한 입력 (3, 4)
h = np.maximum(0, x @ mlp.coefs_[0] + mlp.intercepts_[0])    # 은닉층 : (3,4)×(4,8)+(8,) → ReLU
z = h @ mlp.coefs_[1] + mlp.intercepts_[1]                   # 출력층 가중합 : (3,8)×(8,3)+(3,)
manual = np.array([softmax(row) for row in z])               # 소프트맥스

print("직접 계산     :\n", manual.round(4))
print("predict_proba :\n", clf.predict_proba(X_te[:3]).round(4))
print("같은가? :", np.allclose(manual, clf.predict_proba(X_te[:3])), " / 클래스 순서 :", mlp.classes_)

평가용 정확도 : 1.0
직접 계산     :
 [[9.980e-01 1.900e-03 1.000e-04]
 [6.700e-03 9.180e-01 7.530e-02]
 [9.985e-01 1.400e-03 1.000e-04]]
predict_proba :
 [[9.980e-01 1.900e-03 1.000e-04]
 [6.700e-03 9.180e-01 7.530e-02]
 [9.985e-01 1.400e-03 1.000e-04]]
같은가? : True  / 클래스 순서 : ['setosa' 'versicolor' 'virginica']


sklearn 의 신경망도 내부에서는 이 노트의 **행렬 곱 → 활성화 함수 → 행렬 곱 → 소프트맥스**를 그대로 하고 있다. 학습(가중치를 어떻게 찾는가)은 다음 시간의 **손실 함수 · 경사하강법 · 오차역전파**(교재 p.41~).

## 6. 정리

- 층 하나 = $Y = X \cdot W + B$ (행렬 곱 한 번) → 활성화 함수. 데이터가 여러 개여도 같은 코드.
- 출력층 : 회귀 = 항등 · 1개 / 이진 분류 = 시그모이드 · 1개 / 다중 분류 = 소프트맥스 · 클래스 수.
- 소프트맥스는 확률(합 1)로 바꿀 뿐 **순서는 그대로** → 예측 클래스만 필요하면 가장 큰 z 를 고르면 된다(`argmax`).